# SOLARNESIA: Spatial Model & GIS Analytics for National Solar Power Plant (PLTS) Distribution
**Mata Kuliah:** IMT01303501 - Komputer Grafis (3 SKS)  
**Kelompok 6:** Wesley Goeinawan (0706012410011), Stevanus Ivan Santoso (0706012410029), Nicholas Gerwin Mawardji (0706012410015)  
**Program Studi:** Informatika, Universitas Ciputra Surabaya (2026/2027)

---
### Ringkasan Alur Pemrosesan Spasial
1. **Layer 1 (2D MultiPolygons):** Batas daratan kepulauan dan batas administrasi kabupaten/provinsi (Choropleth rata-rata potensi PVOUT nasional).
2. **Layer 2 (0D Points & Buffers):** Titik observasi Global Solar Atlas darat murni (3.802 titik) setelah *Spatial Clip* mengeliminasi 1.160 titik laut.
3. **5 km Hotspot Clustering:** Pengelompokan titik kandidat berdekatan (≤ 5 km) menjadi sentra hotspot terintegrasi (DBSCAN).
4. **Operasi 1 (Buffer Metrik):** Proyeksi EPSG:3857 dengan koreksi $\cos(\text{lat})$, pembentukan *footprint* 3 km `buffer(3000)` dan koridor transmisi 25 km `buffer(25000)` (standar PLN 20 kV).
5. **Operasi 2 (Spatial Join & Overlay):** Menghitung keterjangkauan 3.289 permukiman darat serta mengekstrak daftar kecamatan/kabupaten yang tercakup secara interaktif.

In [1]:
# Cell 0: Setup & Library Imports
import os  # Mengimpor modul OS untuk operasi sistem berkas dan manipulasi path direktori
import numpy as np  # Mengimpor NumPy untuk komputasi matriks, trigonometri, dan array numerik
import pandas as pd  # Mengimpor Pandas untuk manipulasi tabular dan manajemen DataFrame
import geopandas as gpd  # Mengimpor GeoPandas untuk pengolahan layer spasial dan kalkulasi geometri GIS
from sklearn.cluster import DBSCAN  # Mengimpor algoritma DBSCAN untuk pengelompokan spasial titik tapak PLTS
import folium  # Mengimpor Folium untuk rendering visualisasi peta interaktif berbasis Leaflet.js
from folium.plugins import MarkerCluster, MiniMap, Fullscreen, MeasureControl  # Mengimpor plugin Folium untuk klastering, radar mini, fullscreen, dan pengukur jarak

print(f"GeoPandas: {gpd.__version__} | Pandas: {pd.__version__} | Folium: {folium.__version__}")  # Mencetak versi pustaka utama untuk memverifikasi kompatibilitas lingkungan
os.makedirs("outputs", exist_ok=True)  # Membuat folder penyimpanan hasil ekspor jika belum tersedia
print("Output directory ready.")  # Menampilkan konfirmasi kesiapan direktori output

GeoPandas: 1.2.0 | Pandas: 3.0.2 | Folium: 0.20.0
Output directory ready.


In [2]:
# Cell 1: Ingestion & GeoDataFrame Construction
DATA_DIR = "data"  # Menetapkan nama folder sumber data

# 1. Load Solar Observation Points (Global Solar Atlas)
df_raw = pd.read_csv(os.path.join(DATA_DIR, "indonesia_solar_data.csv"))  # Membaca data tabular mentah radiasi surya dan suhu dari file CSV
raw_pts = gpd.GeoDataFrame(  # Mengonversi DataFrame pandas menjadi objek geospasial GeoDataFrame
    df_raw,  # Menggunakan data atribut tabular dari df_raw
    geometry=gpd.points_from_xy(df_raw["longitude"], df_raw["latitude"]),  # Membuat objek geometri Point dari koordinat bujur dan lintang
    crs="EPSG:4326"  # Menetapkan sistem referensi koordinat geografis WGS84 (derajat)
)

# 2. Load Administrative Boundaries (Provinces & Kabupaten)
gdf_prov = gpd.read_file(os.path.join(DATA_DIR, "indonesia_provinces.geojson"))  # Memuat poligon batas wilayah 34 provinsi Indonesia dari GeoJSON
gdf_kab = gpd.read_file(os.path.join(DATA_DIR, "indonesia_kabupaten.geojson"))  # Memuat poligon batas wilayah 514 kabupaten/kota Indonesia dari GeoJSON

# 3. Load Administrative Sub-Districts (Kecamatan BPS - 7.217 Titik)
df_kec = pd.read_csv(os.path.join(DATA_DIR, "indonesia_kecamatan.csv"))  # Membaca data sebaran titik koordinat pusat kecamatan seluruh Indonesia dari BPS
gdf_kec = gpd.GeoDataFrame(  # Mengonversi titik pusat kecamatan menjadi layer geospasial GeoDataFrame
    df_kec,  # Menggunakan atribut kecamatan dari df_kec
    geometry=gpd.points_from_xy(df_kec["Longitude"], df_kec["Latitude"]),  # Mengonstruksi geometri Point dari kolom Longitude dan Latitude
    crs="EPSG:4326"  # Menetapkan sistem referensi koordinat geografis WGS84
)

assert len(raw_pts) == 4962, f"Expected 4962 points, got {len(raw_pts)}"  # Memastikan total titik observasi radiasi surya tepat 4.962 titik
assert raw_pts.crs.to_epsg() == 4326  # Memvalidasi CRS data surya berada pada EPSG:4326
assert len(gdf_kec) == 7217, f"Expected 7217 kecamatan, got {len(gdf_kec)}"  # Memvalidasi jumlah titik kecamatan sesuai standar BPS yaitu 7.217 titik
print(f"Raw Solar Points loaded: {len(raw_pts)} points | CRS: {raw_pts.crs}")  # Menampilkan status jumlah dan CRS titik surya mentah
print(f"Provinces loaded: {len(gdf_prov)} | Kabupaten loaded: {len(gdf_kab)} | Kecamatan loaded: {len(gdf_kec)}")  # Menampilkan status layer administratif wilayah

assert gdf_kec.crs.to_epsg() == 4326  # Memvalidasi CRS titik kecamatan berada pada EPSG:4326
print("Input verification passed.")  # Menampilkan konfirmasi bahwa seluruh verifikasi data masukan berhasil

Raw Solar Points loaded: 4962 points | CRS: EPSG:4326
Provinces loaded: 34 | Kabupaten loaded: 519 | Kecamatan loaded: 7217
Input verification passed.


In [3]:
# Cell 2: Spatial Clip / Land-Masking (Op 0)
# Mengeliminasi titik-titik observasi yang jatuh di perairan laut menggunakan poligon daratan kabupaten
land_sj = gpd.sjoin(raw_pts, gdf_kab[["shapeName", "geometry"]], how="inner", predicate="intersects")  # Melakukan spatial join inner untuk mencari titik yang menabrak daratan kabupaten
land_pts = raw_pts.loc[land_sj.index.unique()].copy()  # Mengambil titik unik daratan dan menyalinnya ke GeoDataFrame baru

ocean_pts_count = len(raw_pts) - len(land_pts)  # Menghitung selisih jumlah titik untuk mengetahui banyaknya titik laut yang tereliminasi
print(f"Total Raw Points       : {len(raw_pts)}")  # Mencetak total titik awal sebelum masking
print(f"Eliminated Ocean Points: {ocean_pts_count} (Dropped via Land-Masking)")  # Mencetak jumlah titik perairan laut yang dibuang
print(f"Pure Land Points       : {len(land_pts)}")  # Mencetak jumlah titik murni yang berada di atas daratan
assert len(land_pts) == 3802, f"Expected 3802 points, got {len(land_pts)}"  # Memvalidasi jumlah titik daratan tepat 3.802 titik sesuai acuan proposal
assert ocean_pts_count == 1160  # Memvalidasi bahwa tepat 1.160 titik laut telah terbuang

# Pembersihan Memori: Hapus variabel mentah yang tidak digunakan lagi untuk melegakan RAM
del df_raw, raw_pts, land_sj  # Menghapus dataframe dan layer mentah dari memori RAM untuk optimasi kinerja
print("Spatial Clip verification passed & memory cleaned.")  # Mengonfirmasi keberhasilan pemotongan spasial dan pembersihan RAM

Total Raw Points       : 4962
Eliminated Ocean Points: 1160 (Dropped via Land-Masking)
Pure Land Points       : 3802
Spatial Clip verification passed & memory cleaned.


In [4]:
# Cell 3: Site Selection & Kandidat PLTS (Layer 2)
# Ambang batas teknis: PVOUT >= Persentil ke-75 (1.430 kWh/kWp) dan Suhu Rata-rata <= 27,5 °C
p75_pvout = 1430.0  # Menetapkan ambang batas radiasi surya minimal berdasarkan persentil ke-75
max_temp = 27.5  # Menetapkan batas atas temperatur lingkungan untuk menjaga efisiensi fotovoltaik

is_plts = (land_pts["pvout"] >= p75_pvout) & (land_pts["temp"] <= max_temp)  # Membuat masker boolean kondisi kelayakan pembangkit
plts_candidates = land_pts[is_plts].copy()  # Menyaring titik daratan yang memenuhi kedua kriteria teknis menjadi kandidat PLTS
plts_candidates["role"] = "PLTS_CANDIDATE"  # Menandai label peran fungsional sebagai tapak pembangkit surya

print(f"Kandidat PLTS Terpilih : {len(plts_candidates)} titik (PVOUT >= {p75_pvout} & Temp <= {max_temp}°C)")  # Menampilkan total titik PLTS lolos seleksi
assert len(plts_candidates) == 513, f"Expected 513 PLTS candidates, got {len(plts_candidates)}"  # Memverifikasi jumlah kandidat tepat 513 titik
print("Site Selection verified.")  # Mengonfirmasi keberhasilan tahap seleksi tapak

Kandidat PLTS Terpilih : 513 titik (PVOUT >= 1430.0 & Temp <= 27.5°C)
Site Selection verified.


In [5]:
# Cell 4: 5 km Proximity Hotspot Clustering (DBSCAN)
# Titik-titik tapak PLTS dalam radius <= 5 km dikelompokkan ke dalam satu sentra klaster energi
plts_3857 = plts_candidates.to_crs("EPSG:3857")  # Mentransformasikan koordinat kandidat PLTS ke proyeksi metrik Web Mercator
coords_m = np.column_stack([plts_3857.geometry.x, plts_3857.geometry.y])  # Menggabungkan koordinat X dan Y (dalam meter) menjadi matriks 2D untuk DBSCAN

db_5k = DBSCAN(eps=5000, min_samples=1, metric="euclidean").fit(coords_m)  # Menjalankan klastering spasial DBSCAN dengan radius jangkauan 5.000 meter
plts_candidates["cluster_id"] = db_5k.labels_  # Menyimpan label nomor klaster numerik ke dalam GeoDataFrame geografis
plts_3857["cluster_id"] = db_5k.labels_  # Menyimpan label nomor klaster numerik ke dalam GeoDataFrame metrik

plts_candidates["hotspot_id"] = [f"HOTSPOT-{cid:03d}" for cid in db_5k.labels_]  # Membuat kode identitas unik string berformat HOTSPOT-XXX
plts_3857["hotspot_id"] = plts_candidates["hotspot_id"]  # Menyinkronkan identitas hotspot ke layer metrik

unique_hotspots = plts_candidates["hotspot_id"].nunique()  # Menghitung total sentra hotspot unik hasil penggabungan klaster
multi_site_hotspots = plts_candidates.groupby("hotspot_id").filter(lambda g: len(g) > 1)["hotspot_id"].nunique()  # Menghitung jumlah hotspot yang menggabungkan lebih dari satu titik tapak

print(f"Total Kandidat PLTS       : {len(plts_candidates)} titik")  # Menampilkan total titik pembangkit tunggal
print(f"Total Hotspot Sentra      : {unique_hotspots} klaster")  # Menampilkan total klaster sentra energi surya
print(f"Hotspot Multi-titik (<=5km): {multi_site_hotspots} klaster terpadu")  # Menampilkan sentra terpadu multi-tapak

assert unique_hotspots == 480, f"Expected 480 hotspots, got {unique_hotspots}"  # Memverifikasi total hotspot tepat 480 klaster
assert multi_site_hotspots == 31  # Memverifikasi jumlah klaster multi-tapak tepat 31 klaster
print("5km Proximity Clustering verified.")  # Mengonfirmasi keberhasilan pengelompokan kedekatan 5 km

Total Kandidat PLTS       : 513 titik
Total Hotspot Sentra      : 480 klaster
Hotspot Multi-titik (<=5km): 31 klaster terpadu
5km Proximity Clustering verified.


In [6]:
# Cell 5: Transformasi CRS & Pembuatan Buffer Metrik (Op 1)
# Transformasi ke EPSG:3857 dengan koreksi distorsi Mercator: R_adj = R / cos(latitude)
kec_3857 = gdf_kec.to_crs("EPSG:3857")  # Memproyeksikan titik-titik permukiman kecamatan ke sistem koordinat metrik EPSG:3857

lats_plts = plts_candidates["latitude"].values  # Mengekstrak array nilai lintang (latitude) masing-masing kandidat PLTS
rad_adj_25k = 25000.0 / np.cos(np.radians(lats_plts))  # Menghitung radius ekuivalen metrik dengan kompensasi distorsi proyeksi Mercator per garis lintang

# Buffer 25 km (Standar Distribusi 20 kV PLN)
plts_3857["buffer_25km"] = [geom.buffer(r) for geom, r in zip(plts_3857.geometry, rad_adj_25k)]  # Membuat geometri poligon lingkaran buffer 25 km untuk setiap sentra pembangkit

# Poligon Union Koridor Distribusi (Hanya 25 km sesuai instruksi dosen & proposal)
union_25km = gpd.GeoSeries(plts_3857["buffer_25km"], crs="EPSG:3857").union_all()  # Menggabungkan seluruh poligon lingkaran buffer menjadi satu geometri kesatuan koridor distribusi
print("Buffer 25 km generated with latitude-scaling compensation.")  # Mengonfirmasi pembuatan buffer koridor transmisi distribusi selesai

Buffer 25 km generated with latitude-scaling compensation.


In [7]:
# Cell 6: Analisis Keterjangkauan Permukiman Spasial (Op 2)
# Evaluasi Riil Tingkat Kecamatan BPS (7.217 Titik Pusat Permukiman)
gdf_kec["covered_25km"] = kec_3857.geometry.within(union_25km)  # Melakukan uji spasial apakah setiap titik pusat kecamatan berada di dalam koridor buffer 25 km

kec_reached_25 = int(gdf_kec["covered_25km"].sum())  # Menghitung jumlah kecamatan yang berhasil terlayani listrik saluran distribusi 20 kV
kec_unreached_25 = len(gdf_kec) - kec_reached_25  # Menghitung jumlah kecamatan yang belum terjangkau (berstatus off-grid)
kec_pct_25 = (kec_reached_25 / len(gdf_kec)) * 100  # Menghitung persentase rasio elektrifikasi kecamatan terhadap total kecamatan Indonesia

print(f"Total Kecamatan BPS              : {len(gdf_kec):,} titik")  # Menampilkan total titik kecamatan nasional
print(f"Kecamatan Terjangkau 25 km       : {kec_reached_25:,} ({kec_pct_25:.1f}%)")  # Menampilkan statistik kecamatan terlayani
print(f"Kecamatan Belum Terjangkau 25 km : {kec_unreached_25:,} ({(kec_unreached_25 / len(gdf_kec)) * 100:.1f}%)")  # Menampilkan statistik kecamatan off-grid

assert kec_reached_25 == 2806, f"Expected 2806, got {kec_reached_25}"  # Memverifikasi jumlah kecamatan terlayani tepat 2.806 titik
assert kec_unreached_25 == 4411, f"Expected 4411, got {kec_unreached_25}"  # Memverifikasi jumlah kecamatan belum terlayani tepat 4.411 titik
assert round(kec_pct_25, 1) == 38.9  # Memverifikasi rasio persentase keterjangkauan tepat 38,9%
print("Kecamatan reach statistics verified.")  # Mengonfirmasi verifikasi statistik keterjangkauan permukiman berhasil

Total Kecamatan BPS              : 7,217 titik
Kecamatan Terjangkau 25 km       : 2,806 (38.9%)
Kecamatan Belum Terjangkau 25 km : 4,411 (61.1%)
Kecamatan reach statistics verified.


In [8]:
# Cell 7: Ekstraksi Cakupan Wilayah Administratif per Titik / Hotspot
# Melakukan spatial join antara buffer 25 km setiap titik PLTS dengan poligon batas kabupaten
kab_3857 = gdf_kab.to_crs("EPSG:3857")  # Memproyeksikan layer poligon kabupaten ke CRS metrik EPSG:3857

print("Pre-calculating 25km administrative reach for each hotspot...")  # Menampilkan pesan proses kalkulasi irisan wilayah per hotspot dimulai

covered_districts = []  # Menyiapkan list penampung ringkasan nama daerah jangkauan
for idx, geom in enumerate(plts_3857["buffer_25km"]):  # Melakukan iterasi untuk setiap poligon buffer 25 km
    # Spatial intersection dengan poligon kabupaten
    intersecting_kab = kab_3857[kab_3857.geometry.intersects(geom)]  # Menyeleksi kabupaten yang beririsan fisik dengan buffer 25 km hotspot
    kab_names = intersecting_kab["shapeName"].tolist()  # Mengekstrak daftar nama kabupaten hasil irisan spasial
    
    if len(kab_names) > 0:  # Jika terdapat kabupaten yang masuk ke dalam radius jangkauan
        if len(kab_names) <= 3:  # Jika jumlah kabupaten maksimal tiga wilayah
            summary = ", ".join(kab_names)  # Menggabungkan nama-nama kabupaten dipisahkan tanda koma
        else:  # Jika wilayah yang terjangkau melebihi tiga kabupaten
            summary = ", ".join(kab_names[:3]) + f" (+{len(kab_names)-3} wilayah lain)"  # Mengambil 3 kabupaten teratas dan menambahkan informasi jumlah sisanya
    else:  # Jika tidak ada irisan administratif darat
        summary = "Wilayah Pesisir / Non-Administratif"  # Menetapkan status default untuk pulau kecil atau perairan pesisir
    
    covered_districts.append(summary)  # Menambahkan hasil ringkasan ke dalam list utama

plts_candidates["covered_areas_25km"] = covered_districts  # Memasukkan data teks wilayah jangkauan ke kolom baru GeoDataFrame kandidat PLTS
print("Pre-calculation complete. Sample coverage:")  # Menampilkan status pra-perhitungan telah selesai
for i in range(min(5, len(plts_candidates))):  # Melakukan looping untuk menampilkan sampel lima hotspot pertama
    row = plts_candidates.iloc[i]  # Mengambil baris data hotspot ke-i
    print(f"  - {row['hotspot_id']} ({row['latitude']:.2f}, {row['longitude']:.2f}): {row['covered_areas_25km']}")  # Mencetak identitas, koordinat, dan kabupaten terlayani

Pre-calculating 25km administrative reach for each hotspot...


Pre-calculation complete. Sample coverage:
  - HOTSPOT-000 (1.38, 124.62): Kota Manado, Kota Tomohon, Minahasa (+1 wilayah lain)
  - HOTSPOT-001 (-9.99, 120.40): Sumba Timur
  - HOTSPOT-002 (-8.38, 118.18): Bima, Dompu
  - HOTSPOT-003 (-10.34, 123.75): Kota Kupang, Kupang
  - HOTSPOT-004 (0.35, 127.66): Halmahera Selatan, Halmahera Tengah, Kota Tidore Kepulauan


In [9]:
# Cell 8: Rangkuman Metrik Tabel 1 (Analisis Keterjangkauan Permukiman Kecamatan BPS)
summary_table1 = pd.DataFrame([  # Membentuk DataFrame ringkasan Tabel 1 untuk pelaporan formal proyek
    {
        "Parameter Jaringan": "Radius Buffer 25 km (Standar Saluran Distribusi 20 kV PLN)",  # Menjelaskan parameter teknis jangkauan transmisi
        "Total Kecamatan": f"{len(gdf_kec):,}".replace(",", "."),  # Memformat total populasi kecamatan nasional dengan format ribuan titik
        "Kecamatan Terjangkau": f"{kec_reached_25:,} ({kec_pct_25:.1f}%)".replace(",", "."),  # Menyajikan jumlah dan persentase kecamatan yang berhasil terjangkau listrik PLTS
        "Kecamatan Off-Grid": f"{kec_unreached_25:,} ({(kec_unreached_25 / len(gdf_kec)) * 100:.1f}%)".replace(",", ".")  # Menyajikan jumlah dan persentase kecamatan yang masih di luar jangkauan
    }
])

print("Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:")  # Mencetak judul tabel ringkasan
print(summary_table1.to_string(index=False))  # Menampilkan isi tabel ke konsol tanpa indeks numerik pandas

summary_table1  # Mengembalikan objek DataFrame agar dirender sebagai tabel visual di output notebook

Tabel 1: Hasil Analisis Spatial Join Jangkauan Layanan PLTS:
                                        Parameter Jaringan Total Kecamatan Kecamatan Terjangkau Kecamatan Off-Grid
Radius Buffer 25 km (Standar Saluran Distribusi 20 kV PLN)           7.217        2.806 (38.9%)      4.411 (61.1%)


,Parameter Jaringan,Total Kecamatan,Kecamatan Terjangkau,Kecamatan Off-Grid
0,Radius Buffer 25 km (Standar Saluran Distribus...,7.217,2.806 (38.9%),4.411 (61.1%)


In [10]:
# Cell 9: Peta Spasial Interaktif & Inspeksi Klik Terintegrasi (Folium)
# Mengintegrasikan:
# 1. Layer 1: Koroplet Potensi Surya 34 Provinsi (Choropleth Poligon)
# 2. Layer 2: Tapak Sentra PLTS 5 km & Koridor Transmisi 25 km (Buffer Geometris)
# 3. Layer 2: Permukiman Tingkat Kecamatan (7.217 Titik BPS dengan Marker Clustering)
# 4. Fitur Interaktif Tambahan: Measure Control (Pengukur Jarak), MiniMap Radar, Fullscreen, & Floating HUD

import folium
from folium.plugins import MarkerCluster, MiniMap, Fullscreen, MeasureControl

# Center map on Indonesia (Bali & Java focal area)
center_lat, center_lon = -8.40, 115.20
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles="OpenStreetMap",
    control_scale=True
)

# -------------------------------------------------------------
# 1. Layer 1: Koroplet Potensi Surya Provinsi (34 Provinsi)
# -------------------------------------------------------------
prov_means = gpd.sjoin(land_pts, gdf_prov, how='inner', predicate='intersects').groupby('state')['pvout'].mean()
gdf_prov['mean_pvout'] = gdf_prov['state'].map(prov_means).fillna(land_pts['pvout'].mean()).round(1)

prov_layer = folium.FeatureGroup(name="🗺️ Layer 1: Koroplet Potensi Surya Provinsi (34 Provinsi)", show=False)

def prov_style(feature):
    val = feature['properties'].get('mean_pvout', 1400)
    if val >= 1500:
        c = '#dc2626'
    elif val >= 1420:
        c = '#ea580c'
    elif val >= 1350:
        c = '#f59e0b'
    else:
        c = '#fde047'
    return {'fillColor': c, 'color': '#334155', 'weight': 1.2, 'fillOpacity': 0.35}

folium.GeoJson(
    gdf_prov,
    style_function=prov_style,
    tooltip=folium.GeoJsonTooltip(fields=['state', 'mean_pvout'], aliases=['Provinsi:', 'Rata-rata PVOUT (kWh/kWp):'])
).add_to(prov_layer)
prov_layer.add_to(m)

# -------------------------------------------------------------
# 2. Layer 2: Tapak Sentra PLTS 5 km & Koridor Distribusi 25 km
# -------------------------------------------------------------
# Buffer 5 km: Footprint fisik tapak klaster PLTS
hotspot_5k_layer = folium.FeatureGroup(name="☀️ Tapak Sentra PLTS (Footprint 5 km)", show=True)
for idx, row in plts_candidates.iterrows():
    folium.Circle(
        location=[row['latitude'], row['longitude']],
        radius=5000,
        color="#f59e0b",
        weight=2,
        fill=True,
        fill_color="#f59e0b",
        fill_opacity=0.35,
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Footprint Sentra:</b> 5 km"
    ).add_to(hotspot_5k_layer)
hotspot_5k_layer.add_to(m)

# Buffer 25 km: Batas teknis keandalan transmisi TM 20 kV PLN
buffer_25k_layer = folium.FeatureGroup(name="⚡ Koridor Distribusi 25 km (PLN 20 kV)", show=True)
for idx, row in plts_candidates.iterrows():
    folium.Circle(
        location=[row['latitude'], row['longitude']],
        radius=25000,
        color="#38bdf8",
        weight=1.5,
        fill=True,
        fill_color="#38bdf8",
        fill_opacity=0.12,
        tooltip=f"<b>Hotspot:</b> {row['hotspot_id']}<br><b>Radius Distribusi:</b> 25 km"
    ).add_to(buffer_25k_layer)
buffer_25k_layer.add_to(m)

# -------------------------------------------------------------
# 3. Layer 2: Permukiman Berbasis Kecamatan (7.217 Titik BPS)
# -------------------------------------------------------------
# Menggunakan gdf_kec hasil komputasi end-to-end (Cell 1 & Cell 6)
kec_gdf = gdf_kec

# A. Kecamatan Terjangkau 25 km (Emerald Cluster)
kec_reached_cluster = MarkerCluster(name="🟢 Permukiman Kecamatan Terjangkau (2.806 BPS)", show=True)
for idx, row in kec_gdf[kec_gdf['covered_25km']].iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=4,
        color="#059669",
        weight=1.5,
        fill=True,
        fill_color="#10b981",
        fill_opacity=0.9,
        tooltip=f"<b>🏛️ Kec. {row['Name']}</b> ({row['Kabupaten']})<br><span style='color:#059669; font-weight:bold;'>⚡ Terjangkau Feeder 25 km</span>"
    ).add_to(kec_reached_cluster)
kec_reached_cluster.add_to(m)

# B. Kecamatan Belum Terjangkau / Off-Grid (Slate Cluster - Toggleable)
kec_unreached_cluster = MarkerCluster(name="⚪ Permukiman Kecamatan Off-Grid (4.411 BPS)", show=False)
for idx, row in kec_gdf[~kec_gdf['covered_25km']].iterrows():
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=3,
        color="#64748b",
        weight=1,
        fill=True,
        fill_color="#94a3b8",
        fill_opacity=0.6,
        tooltip=f"<b>🏛️ Kec. {row['Name']}</b> ({row['Kabupaten']})<br><span style='color:#dc2626;'>⚪ Di luar Jangkauan 25 km (Off-Grid)</span>"
    ).add_to(kec_unreached_cluster)
kec_unreached_cluster.add_to(m)

# -------------------------------------------------------------
# 4. Layer 2: Kandidat PLTS dengan Popup Inspeksi Klik
# -------------------------------------------------------------
plts_layer = folium.FeatureGroup(name="☀️ Titik Kandidat PLTS (513 Titik)", show=True)
for idx, row in plts_candidates.iterrows():
    popup_html = f"""
    <div style='font-family: Arial, sans-serif; font-size: 12px; width: 270px; line-height: 1.5;'>
        <b style='color: #d97706; font-size: 13px;'>☀️ {row['hotspot_id']}</b><br>
        <b>Koordinat:</b> {row['latitude']:.4f}° LS, {row['longitude']:.4f}° BT<br>
        <b>Potensi PVOUT:</b> <span style='color: #059669; font-weight: bold;'>{row['pvout']:.1f} kWh/kWp</span><br>
        <b>Suhu Rata-rata:</b> {row['temp']:.1f} °C<br>
        <hr style='margin: 6px 0; border: 0; border-top: 1px solid #e5e7eb;'>
        <b>Wilayah Terjangkau (Radius 25 km):</b><br>
        <span style='color: #1e3a8a; font-size: 11px;'>{row['covered_areas_25km']}</span>
    </div>
    """
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6.5,
        color="#ffffff",
        weight=2,
        fill=True,
        fill_color="#f59e0b",
        fill_opacity=0.95,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"<b>Klik untuk Detail:</b> {row['hotspot_id']} (PVOUT: {row['pvout']:.0f})"
    ).add_to(plts_layer)
plts_layer.add_to(m)

# -------------------------------------------------------------
# 5. Floating Glassmorphism HUD Widget
# -------------------------------------------------------------
hud_html = """
<div style="
    position: fixed;
    bottom: 25px;
    left: 25px;
    z-index: 1000;
    background: rgba(15, 23, 42, 0.90);
    backdrop-filter: blur(8px);
    border: 1px solid rgba(56, 189, 248, 0.35);
    border-radius: 12px;
    padding: 12px 18px;
    color: #f8fafc;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 12px;
    box-shadow: 0 10px 25px rgba(0,0,0,0.45);
    max-width: 300px;
    line-height: 1.5;
">
    <div style="display: flex; align-items: center; justify-content: space-between;">
        <b style="color: #38bdf8; font-size: 13px;">☀️ SOLARNESIA GIS HUD</b>
        <span style="font-size: 10px; background: #0369a1; padding: 2px 6px; border-radius: 4px; color: white;">LIVE</span>
    </div>
    <hr style="margin: 7px 0; border: 0; border-top: 1px solid rgba(255,255,255,0.15);">
    <div>⚡ <b>Kandidat PLTS:</b> 513 Titik (480 Hotspot)</div>
    <div>🟡 <b>Footprint Sentra:</b> 5 km</div>
    <div>🔵 <b>Batas Feeder:</b> 25 km (PLN 20 kV)</div>
    <div style="margin-top: 5px; color: #34d399;">🟢 <b>Kecamatan Terjangkau:</b> 2.806 (38,9%)</div>
    <div style="color: #94a3b8;">⚪ <b>Kecamatan Off-Grid:</b> 4.411 (61,1%)</div>
    <div style="margin-top: 6px; font-size: 10.5px; color: #94a3b8; font-style: italic;">
        💡 Gunakan menu layer kanan atas untuk eksplorasi.
    </div>
</div>
"""
m.get_root().html.add_child(folium.Element(hud_html))

# -------------------------------------------------------------
# 6. Kontrol Navigasi & Alat GIS Interaktif
# -------------------------------------------------------------
Fullscreen(position='topleft').add_to(m)
MeasureControl(position='topleft', primary_length_unit='kilometers').add_to(m)
MiniMap(toggle_display=True, position='bottomright').add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
print("Interactive Folium map generated successfully.")
m

Interactive Folium map generated successfully.
